In [2]:
from concurrent.futures import ProcessPoolExecutor

pool = ProcessPoolExecutor()

def p_fib(n):
    if n <= 1:
        return n

    future_x = pool.submit(p_fib, n - 1)
    y = p_fib(n - 2)
    x = future_x.result()

    return x + y

In [1]:
"""
Sequential
"""
def mat_vec(A, x):
    n = len(A)
    y = [0] * n

    for i in range(n):
        for j in range(n):
            y[i] += A[i][j] * x[j]

    return y

"""
Parallel
"""
from concurrent.futures import ProcessPoolExecutor

def row_dot(args):
    row, x = args
    return sum(row[j] * x[j] for j in range(len(x)))

def p_mat_vec(A, x):
    with ProcessPoolExecutor() as executor:
        y = list(executor.map(row_dot, [(row, x) for row in A]))
    return y

In [3]:
from concurrent.futures import ProcessPoolExecutor
import multiprocessing

def compute_row(A, x, i):
    total = 0
    for j in range(len(x)):
        total += A[i][j] * x[j]
    return i, total

def p_mat_recursive(A, x, y, i, I, executor):
    if i == I:
        row_index, value = compute_row(A, x, i)
        y[row_index] = value
        return

    mid = (i + I) // 2

    # spawn left half
    future = executor.submit(
        p_mat_recursive,
        A, x, y,
        i, mid,
        executor
    )

    p_mat_recursive(A, x, y, mid + 1, I, executor)
    future.result()

if __name__ == "__main__":
    n = 4

    A = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16]
    ]

    x = [1, 1, 1, 1]

    # Shared output vector
    manager = multiprocessing.Manager()
    y = manager.list([0] * n)

    with ProcessPoolExecutor() as executor:
        p_mat_recursive(A, x, y, 0, n - 1, executor)

    print(list(y))

In [ ]:
"""
Simplier numpy example:
"""
import numpy as np

A = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
])

x = np.array([1, 1, 1, 1])

y = A @ x

print(y)

In [4]:
"""
T_P = T_1 / P + T_∞

T_P = running time on P processors
T_1 = work (total time on 1 processor)
P = number of processors
T_∞ = span / critical-path length (minimum possible execution time with infinitely many processors)

T_1 = 15
P = 3
T_∞ = 5

T_P ≤ 15/3 + 5
T_P ≤ 5 + 5
T_P ≤ 10

The work term: T_1 / P

If total work is 15 and you have 3 processors:
15 / 3 = 5

meaning each processor ideally does 5 units of work.

The span term: T_∞

This is the unavoidable sequential dependency chain.
Even with infinitely many processors, you still cannot go faster than the longest chain of dependent operations.
For p_fib(5):

fib(5) → fib(4) → fib(3) → fib(2) → fib(1)
which gives span 5.

Running Time ≈ (work per processor) + (unavoidable sequential time)
"""

'\nT_P = T_1 / P + T_∞\n\nT_P = running time on P processors\nT_1 = work (total time on 1 processor)\nP = number of processors\nT_∞ = span / critical-path length (minimum possible execution time with infinitely many processors)\n\nT_1 = 15\nP = 3\nT_∞ = 5\n\nT_P ≤ 15/3 + 5\nT_P ≤ 5 + 5\nT_P ≤ 10\n\nThe work term: T_1 / P\n\nIf total work is 15 and you have 3 processors:\n15 / 3 = 5\n\nmeaning each processor ideally does 5 units of work.\n\nThe span term: T_∞\n\nThis is the unavoidable sequential dependency chain.\nEven with infinitely many processors, you still cannot go faster than the longest chain of dependent operations.\nFor p_fib(5):\n\nfib(5) → fib(4) → fib(3) → fib(2) → fib(1)\nwhich gives span 5.\n\nRunning Time ≈ (work per processor) + (unavoidable sequential time)\n'

In [5]:
"""
In a greedy schedule:

if at least P strands are ready, all processors stay busy,
otherwise, fewer than P strands are ready.

The key observation:
Any step with fewer than P ready strands must execute a strand on the critical path.
Because if a critical-path strand were not executed, then at least one processor could execute it, contradicting greediness.

Thus:
every "incomplete" step reduces the remaining span by 1,
and there can be at most T_∞ such steps.

The standard bound:
T_P ≤ T_1/P + T_∞
double-counts some work on the critical path.

The stronger bound removes that overlap:
T_P ≤ (T_1 - T_∞)/P + T_∞

T_∞ units of work are already "reserved" for the critical path,
only the remaining work must be divided among processors.

So the improved bound is slightly tighter.

======================================================================================================================================

A "good" greedy scheduler always advances the critical path immediately.
A "bad" greedy scheduler repeatedly uses processors on side work first, delaying critical-path progress by almost one extra step per level.

Fast schedule ≈ T∞
Slow schedule ≈ 2T∞

Greedy scheduling guarantees processors never sit idle unnecessarily, but it does not guarantee that the scheduler chooses the "best" ready strands.
"""

'\nIn a greedy schedule:\n\nif at least P strands are ready, all processors stay busy,\notherwise, fewer than P strands are ready.\n\nThe key observation:\nAny step with fewer than P ready strands must execute a strand on the critical path.\nBecause if a critical-path strand were not executed, then at least one processor could execute it, contradicting greediness.\n\nThus:\nevery "incomplete" step reduces the remaining span by 1,\nand there can be at most T_∞ such steps.\n\nThe standard bound:\nT_P ≤ T_1/P + T_∞\ndouble-counts some work on the critical path.\n\nThe stronger bound removes that overlap:\nT_P ≤ (T_1 - T_∞)/P + T_∞\n\nT_∞ units of work are already "reserved" for the critical path,\nonly the remaining work must be divided among processors.\n\nSo the improved bound is slightly tighter.\n\n======================================================================================================================================\n\nA "good" greedy scheduler always advances the criti

In [6]:
"""
Given measurements:

T_4 = 80
T_10 = 42
T_64 = 10

Meaning:

T_4 = 80
→ the program took 80 seconds when executed on 4 processors.

T_10 = 42
→ the program took 42 seconds when executed on 10 processors.

T_64 = 10
→ the program took 10 seconds when executed on 64 processors.

These values are execution times measured for the SAME parallel algorithm using different numbers of processors. As the number of processors increases, the running time should generally decrease because more work can be done simultaneously.

Using the Work Law:

T_P ≥ T_1 / P

we estimate the total work T_1. Using the 4-processor run:

80 ≥ T_1 / 4

which gives:

T_1 ≤ 320

This means the total amount of work done by the algorithm is at most 320 units of time.

Using the Span Law:

T_P ≥ T_∞

and the 64-processor run:

10 ≥ T_∞

which gives:

T_∞ ≤ 10

This means the critical path (the longest chain of dependent operations that cannot run in parallel) is at most 10 time units.

Finally, using the greedy scheduler bound:

T_P ≤ (T_1 - T_∞)/P + T_∞

for P = 10:

T_10 ≤ (320 - 10)/10 + 10
T_10 ≤ 31 + 10
T_10 ≤ 41

But the measurements claim:

T_10 = 42

which violates the theoretical upper bound. Therefore, the three timing measurements cannot all be correct simultaneously.
"""

'\nGiven measurements:\n\nT_4 = 80\nT_10 = 42\nT_64 = 10\n\nMeaning:\n\nT_4 = 80\n→ the program took 80 seconds when executed on 4 processors.\n\nT_10 = 42\n→ the program took 42 seconds when executed on 10 processors.\n\nT_64 = 10\n→ the program took 10 seconds when executed on 64 processors.\n\nThese values are execution times measured for the SAME parallel algorithm using different numbers of processors. As the number of processors increases, the running time should generally decrease because more work can be done simultaneously.\n\nUsing the Work Law:\n\nT_P ≥ T_1 / P\n\nwe estimate the total work T_1. Using the 4-processor run:\n\n80 ≥ T_1 / 4\n\nwhich gives:\n\nT_1 ≤ 320\n\nThis means the total amount of work done by the algorithm is at most 320 units of time.\n\nUsing the Span Law:\n\nT_P ≥ T_∞\n\nand the 64-processor run:\n\n10 ≥ T_∞\n\nwhich gives:\n\nT_∞ ≤ 10\n\nThis means the critical path (the longest chain of dependent operations that cannot run in parallel) is at most 10 

In [7]:
"""
parallel for i = 1 to n
    parallel for j = 1 to n
        p[j] = A[i][j] * x[j]

    y[i] = parallel_reduce_sum(p[1..n])

T_1/T_∞ = Θ(n^2)/Θ(lgn)

Parallelism = Θ(n^2 / lg n)

| Quantity    | Value         |
| ----------- | --------------|
| Work        | Θ(n^2)        |
| Span        | Θ(lgn)        |
| Parallelism | Θ(n^2 / lg n) |

========================================================================================================

Given the modified algorithm:
def p_transpose(A, n):
    for j = 2 to n
        parallel for i = 1 to j-1
            exchange A[i][j] with A[j][i]

the total work T_1 is unchanged because the algorithm still performs the same number of swaps. The total number of exchanges is:

Σ(j−1) from j=2 to n = 1 + 2 + ... + (n−1) = n(n−1)/2

therefore:

T_1 = Θ(n²)

The span changes because the outer loop is now serial. For each value of j, the inner parallel loop has span Θ(lg j). Since the outer loop executes sequentially, the spans add together:

T_∞ = ΣΘ(lg j) from j=2 to n

which simplifies using logarithm summation:

T_∞ = Θ(lg(n!)) = Θ(n lg n)

Finally, the parallelism is:

T_1 / T_∞ = Θ(n²) / Θ(n lg n) = Θ(n / lg n)

Therefore, the modified algorithm has:
Work: T_1 = Θ(n²)
Span: T_∞ = Θ(n lg n)
Parallelism: Θ(n / lg n)

The key difference is that only the inner loop remains parallel, while the outer loop forces the computation to proceed column-by-column sequentially, increasing the critical-path length substantially.

========================================================================================================

For the fully parallel version of p_transpose, the running time is:
T_P^(1) = n²/P + lg n

because the work is T_1 = Θ(n²) and the span is T_∞ = Θ(lg n). For the modified version with a serial outer loop, the running time is:
T_P^(2) = n²/P + n lg n

because the work remains T_1 = Θ(n²) but the span increases to T_∞ = Θ(n lg n). To determine when the two algorithms run equally fast, set the running times equal:
n²/P + lg n = n²/P + n lg n

Subtracting n²/P from both sides gives:
lg n = n lg n

Dividing both sides by lg n yields:
1 = n

Therefore, the two versions are equally fast only when n = 1. For any practical matrix size n > 1, the fully parallel version is always faster because both algorithms perform the same total work, but the second version has a much larger span, which increases the critical-path length and limits parallel speedup.
"""

'\nparallel for i = 1 to n\n    parallel for j = 1 to n\n        p[j] = A[i][j] * x[j]\n\n    y[i] = parallel_reduce_sum(p[1..n])\n\nT_1/T_∞ = Θ(n^2)/Θ(lgn)\n\nParallelism = Θ(n^2 / lg n)\n\n| Quantity    | Value         |\n| ----------- | --------------|\n| Work        | Θ(n^2)        |\n| Span        | Θ(lgn)        |\n| Parallelism | Θ(n^2 / lg n) |\n\n========================================================================================================\n\nGiven the modified algorithm:\ndef p_transpose(A, n):\n    for j = 2 to n\n        parallel for i = 1 to j-1\n            exchange A[i][j] with A[j][i]\n\nthe total work T_1 is unchanged because the algorithm still performs the same number of swaps. The total number of exchanges is:\n\nΣ(j−1) from j=2 to n = 1 + 2 + ... + (n−1) = n(n−1)/2\n\ntherefore:\n\nT_1 = Θ(n²)\n\nThe span changes because the outer loop is now serial. For each value of j, the inner parallel loop has span Θ(lg j). Since the outer loop executes sequentially,

In [9]:
"""
sequential version
"""
def zero_matrix(n):
    return [[0 for _ in range(n)] for _ in range(n)]

def add_matrix(A, B):
    n = len(A)
    return [
        [A[i][j] + B[i][j] for j in range(n)]
        for i in range(n)
    ]

def split_matrix(M):
    n = len(M)
    mid = n // 2

    A11 = [row[:mid] for row in M[:mid]]
    A12 = [row[mid:] for row in M[:mid]]
    A21 = [row[:mid] for row in M[mid:]]
    A22 = [row[mid:] for row in M[mid:]]

    return A11, A12, A21, A22

def combine_matrix(C11, C12, C21, C22):
    top = [C11[i] + C12[i] for i in range(len(C11))]
    bottom = [C21[i] + C22[i] for i in range(len(C21))]
    return top + bottom

def recursive_multiply(A, B):
    n = len(A)

    if n == 1:
        return [[A[0][0] * B[0][0]]]

    A11, A12, A21, A22 = split_matrix(A)
    B11, B12, B21, B22 = split_matrix(B)

    C11 = add_matrix(
        recursive_multiply(A11, B11),
        recursive_multiply(A12, B21)
    )

    C12 = add_matrix(
        recursive_multiply(A11, B12),
        recursive_multiply(A12, B22)
    )

    C21 = add_matrix(
        recursive_multiply(A21, B11),
        recursive_multiply(A22, B21)
    )

    C22 = add_matrix(
        recursive_multiply(A21, B12),
        recursive_multiply(A22, B22)
    )

    return combine_matrix(C11, C12, C21, C22)


A = [
    [1, 2],
    [3, 4]
]

B = [
    [5, 6],
    [7, 8]
]

C = recursive_multiply(A, B)

for row in C:
    print(row)

[19, 22]
[43, 50]


In [ ]:
"""
parallel (without numpy)

even with numpy we still have GIL limitations,
best parallel outcome would use different language like C/RUST etc
"""
from concurrent.futures import ProcessPoolExecutor

def zero_matrix(n):
    return [[0 for _ in range(n)] for _ in range(n)]

def add_matrix(A, B):
    n = len(A)
    return [
        [A[i][j] + B[i][j] for j in range(n)]
        for i in range(n)
    ]

def split_matrix(M):
    n = len(M)
    mid = n // 2

    M11 = [row[:mid] for row in M[:mid]]
    M12 = [row[mid:] for row in M[:mid]]
    M21 = [row[:mid] for row in M[mid:]]
    M22 = [row[mid:] for row in M[mid:]]

    return M11, M12, M21, M22

def combine_matrix(C11, C12, C21, C22):
    top = [C11[i] + C12[i] for i in range(len(C11))]
    bottom = [C21[i] + C22[i] for i in range(len(C21))]
    return top + bottom

def recursive_multiply(A, B):
    n = len(A)

    if n == 1:
        return [[A[0][0] * B[0][0]]]

    A11, A12, A21, A22 = split_matrix(A)
    B11, B12, B21, B22 = split_matrix(B)

    C11 = add_matrix(
        recursive_multiply(A11, B11),
        recursive_multiply(A12, B21)
    )

    C12 = add_matrix(
        recursive_multiply(A11, B12),
        recursive_multiply(A12, B22)
    )

    C21 = add_matrix(
        recursive_multiply(A21, B11),
        recursive_multiply(A22, B21)
    )

    C22 = add_matrix(
        recursive_multiply(A21, B12),
        recursive_multiply(A22, B22)
    )

    return combine_matrix(C11, C12, C21, C22)

def parallel_matrix_multiply(A, B):
    A11, A12, A21, A22 = split_matrix(A)
    B11, B12, B21, B22 = split_matrix(B)

    with ProcessPoolExecutor() as executor:
        f1 = executor.submit(recursive_multiply, A11, B11)
        f2 = executor.submit(recursive_multiply, A12, B21)

        f3 = executor.submit(recursive_multiply, A11, B12)
        f4 = executor.submit(recursive_multiply, A12, B22)

        f5 = executor.submit(recursive_multiply, A21, B11)
        f6 = executor.submit(recursive_multiply, A22, B21)

        f7 = executor.submit(recursive_multiply, A21, B12)
        f8 = executor.submit(recursive_multiply, A22, B22)

        C11 = add_matrix(f1.result(), f2.result())
        C12 = add_matrix(f3.result(), f4.result())
        C21 = add_matrix(f5.result(), f6.result())
        C22 = add_matrix(f7.result(), f8.result())

    return combine_matrix(C11, C12, C21, C22)

if __name__ == "__main__":

    A = [
        [1, 2],
        [3, 4]
    ]

    B = [
        [5, 6],
        [7, 8]
    ]

    C = parallel_matrix_multiply(A, B)

    print("Result:")

    for row in C:
        print(row)

In [8]:
import numpy as np

def p_matrix_multiply_recursive(A, B, C, n):
    if n == 1:
        C[0, 0] += A[0, 0] * B[0, 0]
        return

    # Temporary matrix D
    D = np.zeros((n, n), dtype=A.dtype)

    mid = n // 2

    # Partition matrices into quadrants
    A11 = A[:mid, :mid]
    A12 = A[:mid, mid:]
    A21 = A[mid:, :mid]
    A22 = A[mid:, mid:]

    B11 = B[:mid, :mid]
    B12 = B[:mid, mid:]
    B21 = B[mid:, :mid]
    B22 = B[mid:, mid:]

    C11 = C[:mid, :mid]
    C12 = C[:mid, mid:]
    C21 = C[mid:, :mid]
    C22 = C[mid:, mid:]

    D11 = D[:mid, :mid]
    D12 = D[:mid, mid:]
    D21 = D[mid:, :mid]
    D22 = D[mid:, mid:]

    p_matrix_multiply_recursive(A11, B11, C11, mid)
    p_matrix_multiply_recursive(A11, B12, C12, mid)
    p_matrix_multiply_recursive(A21, B11, C21, mid)
    p_matrix_multiply_recursive(A21, B12, C22, mid)

    p_matrix_multiply_recursive(A12, B21, D11, mid)
    p_matrix_multiply_recursive(A12, B22, D12, mid)
    p_matrix_multiply_recursive(A22, B21, D21, mid)
    p_matrix_multiply_recursive(A22, B22, D22, mid)

    C += D

A = np.array([[1, 2],
              [3, 4]])

B = np.array([[5, 6],
              [7, 8]])

n = A.shape[0]
C = np.zeros((n, n), dtype=int)

p_matrix_multiply_recursive(A, B, C, n)

In [10]:
"""
p_matrix_multiply (2×2)

Level 0 (spawn)

          Start
             |
 -------------------------------------------------
 |   |   |   |   |   |   |   |
 M1  M2  M3  M4  M5  M6  M7  M8
 |   |   |   |   |   |   |   |
 -------------------------------------------------
             |
           Sync
             |
 -----------------------------------------
 |          |          |          |
 A1         A2         A3         A4
             |
            End

For 2×2 matrices:

| Quantity              | Value |
| --------------------- | ----- |
| Work (T_1)            | 12    |
| Span (T_∞)            | 2     |
| Parallelism (T_1/T_∞) | 6     |

For 2x2 matrices:
C11 = A11*B11 + A12*B21
C12 = A11*B12 + A12*B22
C21 = A21*B11 + A22*B21
C22 = A21*B12 + A22*B22

Spawned multiplication strands:
M1 = A11*B11
M2 = A11*B12
M3 = A21*B11
M4 = A21*B12
M5 = A12*B21
M6 = A12*B22
M7 = A22*B21
M8 = A22*B22

Addition strands after sync:
A1 = C11 += D11
A2 = C12 += D12
A3 = C21 += D21
A4 = C22 += D22


Work Analysis:
T1 = 8 multiplications + 4 additions
T1 = 12

Work = Theta(1) for fixed 2x2 case


Span Analysis:

Critical path:

Start
-> one multiplication
-> sync
-> one addition
-> End

T_infinity = 2

Span = Theta(1)

Parallelism:
Parallelism = T1 / T_infinity
Parallelism = 12 / 2
Parallelism = 6


General Recursive Case:
Work recurrence:
T1(n) = 8T1(n/2) + Theta(n^2)

Using Master Theorem:
T1(n) = Theta(n^3)

Span recurrence:
T_infinity(n) = T_infinity(n/2) + Theta(log n)

Therefore:
T_infinity(n) = Theta(log^2 n)


Parallelism:

Parallelism(n) =
T1(n) / T_infinity(n)

= Theta(n^3 / log^2 n)
"""

'\np_matrix_multiply (2×2)\n\nLevel 0 (spawn)\n\n          Start\n             |\n -------------------------------------------------\n |   |   |   |   |   |   |   |\n M1  M2  M3  M4  M5  M6  M7  M8\n |   |   |   |   |   |   |   |\n -------------------------------------------------\n             |\n           Sync\n             |\n -----------------------------------------\n |          |          |          |\n A1         A2         A3         A4\n             |\n            End\n\nFor 2×2 matrices:\n\n| Quantity              | Value |\n| --------------------- | ----- |\n| Work (T_1)            | 12    |\n| Span (T_∞)            | 2     |\n| Parallelism (T_1/T_∞) | 6     |\n\nFor 2x2 matrices:\nC11 = A11*B11 + A12*B21\nC12 = A11*B12 + A12*B22\nC21 = A21*B11 + A22*B21\nC22 = A21*B12 + A22*B22\n\nSpawned multiplication strands:\nM1 = A11*B11\nM2 = A11*B12\nM3 = A21*B11\nM4 = A21*B12\nM5 = A12*B21\nM6 = A12*B22\nM7 = A22*B21\nM8 = A22*B22\n\nAddition strands after sync:\nA1 = C11 += D11\nA

In [11]:
"""
p_matrix_multiply_recursive(A, B, C, n)

Base Case:

if n == 1:
    C[0][0] += A[0][0] * B[0][0]

Recursive Computation:
Partition matrices into quadrants:

A =
| A11  A12 |
| A21  A22 |

B =
| B11  B12 |
| B21  B22 |

C =
| C11  C12 |
| C21  C22 |

D =
| D11  D12 |
| D21  D22 |

Spawned Recursive Multiplications:

1. p_matrix_multiply_recursive(A11, B11, C11, n/2)
2. p_matrix_multiply_recursive(A11, B12, C12, n/2)
3. p_matrix_multiply_recursive(A21, B11, C21, n/2)
4. p_matrix_multiply_recursive(A21, B12, C22, n/2)

5. p_matrix_multiply_recursive(A12, B21, D11, n/2)
6. p_matrix_multiply_recursive(A12, B22, D12, n/2)
7. p_matrix_multiply_recursive(A22, B21, D21, n/2)
8. p_matrix_multiply_recursive(A22, B22, D22, n/2)


Sync:
wait for all 8 recursive multiplications to complete

Final Parallel Addition:
C11 += D11
C12 += D12
C21 += D21
C22 += D22

--------------------------------------------------
WORK ANALYSIS
--------------------------------------------------

Work recurrence:
T1(n) = 8T1(n/2) + Theta(n^2)

Explanation:
- 8 recursive multiplications
- Theta(n^2) work for matrix additions

Using Master Theorem:

a = 8
b = 2

n^(log_b(a))
= n^(log_2(8))
= n^3

f(n) = Theta(n^2)

Since:
Theta(n^2) = O(n^(3-epsilon))

Case 1 of Master Theorem applies.

Therefore:
T1(n) = Theta(n^3)

--------------------------------------------------
SPAN ANALYSIS
--------------------------------------------------

Since the 8 recursive calls execute in parallel,
only ONE recursive branch contributes to the span.

Span recurrence:

T_∞(n)
= T_∞(n/2) + Theta(log n)

Explanation:
- recursive dependency chain
- parallel matrix addition costs Theta(log n)


Solving recurrence:
T_∞(n) = Theta(log^2 n)

--------------------------------------------------
PARALLELISM
--------------------------------------------------

Parallelism(n)
=
T1(n) / T_∞(n)

=
Theta(n^3 / log^2 n)

--------------------------------------------------
2x2 MATRIX CASE
--------------------------------------------------

For n = 2:
Recursive calls immediately hit base case.

Total strands:
- 8 multiplication strands
- 4 addition strands

Work:
T1 = 12

Critical path:
Start
-> one multiplication
-> sync
-> one addition
-> End

Span:
T_∞ = 2

Parallelism:
12 / 2 = 6
"""

'\np_matrix_multiply_recursive(A, B, C, n)\n\nBase Case:\n\nif n == 1:\n    C[0][0] += A[0][0] * B[0][0]\n\nRecursive Computation:\nPartition matrices into quadrants:\n\nA =\n| A11  A12 |\n| A21  A22 |\n\nB =\n| B11  B12 |\n| B21  B22 |\n\nC =\n| C11  C12 |\n| C21  C22 |\n\nD =\n| D11  D12 |\n| D21  D22 |\n\nSpawned Recursive Multiplications:\n\n1. p_matrix_multiply_recursive(A11, B11, C11, n/2)\n2. p_matrix_multiply_recursive(A11, B12, C12, n/2)\n3. p_matrix_multiply_recursive(A21, B11, C21, n/2)\n4. p_matrix_multiply_recursive(A21, B12, C22, n/2)\n\n5. p_matrix_multiply_recursive(A12, B21, D11, n/2)\n6. p_matrix_multiply_recursive(A12, B22, D12, n/2)\n7. p_matrix_multiply_recursive(A22, B21, D21, n/2)\n8. p_matrix_multiply_recursive(A22, B22, D22, n/2)\n\n\nSync:\nwait for all 8 recursive multiplications to complete\n\nFinal Parallel Addition:\nC11 += D11\nC12 += D12\nC21 += D21\nC22 += D22\n\n--------------------------------------------------\nWORK ANALYSIS\n------------------------

In [ ]:
"""
ThreadPoolExecutor does NOT help CPU-heavy math much because of the GIL.
For true parallel CPU execution, use:
ProcessPoolExecutor
NumPy
CUDA/OpenMP/etc.
"""
from concurrent.futures import ThreadPoolExecutor
import math

def parallel_sum(arr):
    arr = arr[:]

    while len(arr) > 1:
        next_level = [0] * math.ceil(len(arr) / 2)

        def add_pair(i):
            left = arr[2 * i]

            # Handle odd length
            if 2 * i + 1 < len(arr):
                right = arr[2 * i + 1]
                return left + right

            return left

        with ThreadPoolExecutor() as executor:
            results = executor.map(
                add_pair,
                range(len(next_level))
            )

            next_level = list(results)

        arr = next_level
        print("Next reduction level:", arr)

    return arr[0]


T = [1, 2, 3, 4, 5, 6, 7, 8]
result = parallel_sum(T)
print("Final Sum:", result)

In [12]:
"""
Θ(n)

Sequential Sum:
Span = Theta(n)

example:
sum = 0
for i in range(n):
    sum += T[i]

=================================================
T∞ = Θ(logn)

Instead of Sequential (parallel):
((((a+b)+c)+d)+e)

Think of parallel sum as:
we do additions simultaneously:

Level 1:
(a+b)   (c+d)   (e+f)   (g+h)

Level 2:
((a+b)+(c+d))   ((e+f)+(g+h))

Level 3:
final sum

example:
T = [1,2,3,4,5,6,7,8]

1+2 = 3
3+4 = 7
5+6 = 11
7+8 = 15
T = [3,7,11,15]

Round 2:
3+7 = 10
11+15 = 26
T = [10,26]

Round 3:
10+26 = 36

Next reduction level: [3, 7, 11, 15]
Next reduction level: [10, 26]
Next reduction level: [36]

Final Sum: 36

INTUITION
7 additions in a chain

Parallel reduction:
log2(8) = 3 levels

T[i] = T[2*i] + T[2*i + 1]
pair neighboring elements together

| i | computes  |
| - | --------- |
| 0 | T[0]+T[1] |
| 1 | T[2]+T[3] |
| 2 | T[4]+T[5] |

Each level runs in parallel.
                36
              /    \
            10      26
           /  \    /  \
          3    7  11  15
         /\   /\  /\  /\
        1 2  3 4 5 6 7 8
"""

'\nΘ(n)\n\nSequential Sum:\nSpan = Theta(n)\n\nexample:\nsum = 0\nfor i in range(n):\n    sum += T[i]\n\n=================================================\nT∞ = Θ(logn)\n\nInstead of Sequential (parallel):\n((((a+b)+c)+d)+e)\n\nThink of parallel sum as:\nwe do additions simultaneously:\n\nLevel 1:\n(a+b)   (c+d)   (e+f)   (g+h)\n\nLevel 2:\n((a+b)+(c+d))   ((e+f)+(g+h))\n\nLevel 3:\nfinal sum\n\nexample:\nT = [1,2,3,4,5,6,7,8]\n\n1+2 = 3\n3+4 = 7\n5+6 = 11\n7+8 = 15\nT = [3,7,11,15]\n\nRound 2:\n3+7 = 10\n11+15 = 26\nT = [10,26]\n\nRound 3:\n10+26 = 36\n\nNext reduction level: [3, 7, 11, 15]\nNext reduction level: [10, 26]\nNext reduction level: [36]\n\nFinal Sum: 36\n\nINTUITION\n7 additions in a chain\n\nParallel reduction:\nlog2(8) = 3 levels\n\nT[i] = T[2*i] + T[2*i + 1]\npair neighboring elements together\n\n| i | computes  |\n| - | --------- |\n| 0 | T[0]+T[1] |\n| 1 | T[2]+T[3] |\n| 2 | T[4]+T[5] |\n\nEach level runs in parallel.\n                36\n              /             

In [ ]:
"""
Parallel Floyd-Warshall Algorithm

adjacency matrix
dist[i][j] = edge weight
INF if no edge

Better Fully Parallel Version (Theoretical)
If both rows and columns are parallelized:

parallel for i
    parallel for j
    
| Quantity    | Complexity    |
| ----------- | ------------- |
| Work        | (Theta(n^3)) |
| Span        | (Theta(n))   |
| Parallelism | (Theta(n^2)) |
"""
from concurrent.futures import ThreadPoolExecutor
import math

INF = float('inf')

def parallel_floyd_warshall(dist):
    n = len(dist)
    
    for k in range(n):
        # parallel
        def update_row(i):
            
            for j in range(n):
                # Relaxation step
                dist[i][j] = min(
                    dist[i][j],
                    dist[i][k] + dist[k][j]
                )

        with ThreadPoolExecutor() as executor:
            executor.map(update_row, range(n))

    return dist

# --------------------------------------------------
# Example Graph
# --------------------------------------------------

graph = [
    [0,     3,     INF,   7],
    [8,     0,     2,     INF],
    [5,     INF,   0,     1],
    [2,     INF,   INF,   0]
]

result = parallel_floyd_warshall(graph)
print("Shortest Path Matrix:\n")

for row in result:
    print(row)

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def merge_sort(A):
    if len(A) <= 1:
        return A

    mid = len(A) // 2

    left = merge_sort(A[:mid])
    right = merge_sort(A[mid:])

    return merge(left, right)

def merge(left, right):
    result = []

    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])

    return result

def p_merge_sort(A, threshold=1000):
    if len(A) <= 1:
        return A

    # Avoid excessive process overhead
    if len(A) <= threshold:
        return merge_sort(A)

    mid = len(A) // 2

    with ProcessPoolExecutor() as executor:
        left_future = executor.submit(
            p_merge_sort,
            A[:mid],
            threshold
        )

        right_future = executor.submit(
            p_merge_sort,
            A[mid:],
            threshold
        )

        left = left_future.result()
        right = right_future.result()

    return merge(left, right)

In [13]:
def find_split_point(A, p, r, x):
    low = p
    high = r + 1

    while low < high:
        mid = (low + high) // 2
        
        if x <= A[mid]:
            high = mid
        else:
            low = mid + 1

    return low

In [ ]:
# --------------------------------------------------
# Parallel Merge Wrapper
# --------------------------------------------------
def p_merge_with_aux(A, p, q, r):

    # Auxiliary array
    B = [0] * len(A)

    p_merge_aux(A, p, q, q + 1, r, B, p)

    # Copy back
    for i in range(p, r + 1):
        A[i] = B[i]

def p_merge_aux(A, p1, r1, p2, r2, B, p3):
    if p1 > r1 and p2 > r2:
        return

    # Ensure first subarray is larger
    if (r1 - p1) < (r2 - p2):
        p1, p2 = p2, p1
        r1, r2 = r2, r1

    if p1 > r1:
        return

    q1 = (p1 + r1) // 2
    x = A[q1]
    q2 = find_split_point(A, p2, r2, x)
    q3 = p3 + (q1 - p1) + (q2 - p2)
    B[q3] = x

    with ThreadPoolExecutor() as executor:
        left_future = executor.submit(
            p_merge_aux,
            A,
            p1,
            q1 - 1,
            p2,
            q2 - 1,
            B,
            p3
        )
        right_future = executor.submit(
            p_merge_aux,
            A,
            q1 + 1,
            r1,
            q2,
            r2,
            B,
            q3 + 1
        )

        left_future.result()
        right_future.result()

A = [1, 3, 5, 7, 2, 4, 6, 8]

p = 0
q = 3
r = 7

p_merge_with_aux(A, p, q, r)

print(A)

In [14]:
"""
The auxiliary array is extremely important for parallel merge.

Without it, threads/processes would overwrite values in A while other recursive calls are still reading from A.

Read from A
Write into B

Then at the end:
Copy B back into A

no overwriting
no synchronization needed between writes
safe parallelism

This line computes exactly where the pivot goes:
q3 = p3 + (q1 - p1) + (q2 - p2)

Meaning:
how many elements are smaller than x
So each recursive call owns a distinct section of B.
That's what makes the merge parallel-safe.
"""

"\nThe auxiliary array is extremely important for parallel merge.\n\nWithout it, threads/processes would overwrite values in A while other recursive calls are still reading from A.\n\nRead from A\nWrite into B\n\nThen at the end:\nCopy B back into A\n\nno overwriting\nno synchronization needed between writes\nsafe parallelism\n\nThis line computes exactly where the pivot goes:\nq3 = p3 + (q1 - p1) + (q2 - p2)\n\nMeaning:\nhow many elements are smaller than x\nSo each recursive call owns a distinct section of B.\nThat's what makes the merge parallel-safe.\n"

In [16]:
"""
"Coarsening the base case" for p_merge() means:
Increase the threshold at which recursion stops, and use a simple sequential merge for those smaller chunks.

A simplified P-MERGE looks like this:

P-MERGE(A, B):
    if |A| + |B| == 0:
        return

    choose median element x
    partition the other array around x
    recursively merge left halves in parallel
    recursively merge right halves in parallel

Coarsened base case
if |A| + |B| <= k:
    sequential_merge(A, B)
    return

where:

k is a tuning parameter
often between 32 and several thousand
depends on hardware/cache/runtime

Without coarsening:
size 1 → recurse
size 2 → recurse
size 4 → recurse
...

With coarsening:
size 1024 → recurse
size 512 → recurse
size 64 → sequential merge
"""

'\n"Coarsening the base case" for p_merge() means:\nIncrease the threshold at which recursion stops, and use a simple sequential merge for those smaller chunks.\n'

In [ ]:
"""
Comparison with Standard P-MERGE

Standard P-MERGE:
chooses median from one array
binary-searches in the other
guarantees only approximate balance

This modified version:
finds the exact global median
produces nearly perfect n/2 splits
keeps the same asymptotic bounds:
work: O(n)
span: O(log2n)
"""
from concurrent.futures import ThreadPoolExecutor

INF = float("inf")


def kth_smallest(A, B, k):
    m, n = len(A), len(B)

    # Ensure A is smaller
    if m > n:
        value, j, i = kth_smallest(B, A, k)
        return value, i, j

    low = max(0, k - n)
    high = min(k, m)

    while low <= high:
        i = (low + high) // 2
        j = k - i

        Aleft  = -INF if i == 0 else A[i - 1]
        Aright = INF if i == m else A[i]

        Bleft  = -INF if j == 0 else B[j - 1]
        Bright = INF if j == n else B[j]

        if Aleft <= Bright and Bleft <= Aright:
            return max(Aleft, Bleft), i, j

        elif Aleft > Bright:
            high = i - 1
        else:
            low = i + 1

    raise ValueError("Unreachable")

In [ ]:
"""
At each recursive step:

the global median x is found
every element in:
(leftA ∪ leftB) is ≤ x
(rightA ∪ rightB) is ≥ x

Thus:
merge(left) + [x] + merge(right)
"""
def serial_merge(A, B):
    i = j = 0
    out = []

    while i < len(A) and j < len(B):
        if A[i] <= B[j]:
            out.append(A[i])
            i += 1
        else:
            out.append(B[j])
            j += 1

    out.extend(A[i:])
    out.extend(B[j:])

    return out


CUTOFF = 32

"""
Parallel merge using the true median of A ∪ B.
"""
def p_merge_median(A, B, executor=None):
    total = len(A) + len(B)

    # Coarsened base case
    if total <= CUTOFF:
        return serial_merge(A, B)

    k = total // 2

    median, i, j = kth_smallest(A, B, k)

    leftA = A[:i]
    rightA = A[i:]

    leftB = B[:j]
    rightB = B[j:]

    # Remove one copy of the median from the right side
    if rightA and rightA[0] == median:
        rightA = rightA[1:]
    else:
        rightB = rightB[1:]

    if executor is None:
        with ThreadPoolExecutor() as ex:
            left_future = ex.submit(p_merge_median, leftA, leftB, ex)
            right = p_merge_median(rightA, rightB, ex)
            left = left_future.result()
    else:
        left_future = executor.submit(
            p_merge_median, leftA, leftB, executor
        )

        right = p_merge_median(rightA, rightB, executor)
        left = left_future.result()

    return left + [median] + right

In [17]:
"""
This is the standard high-performance parallel partition strategy used in many parallel quicksort implementations.
every element in the left part is ≤ the median
every element in the right part is ≥ the median
the pivot/median ends up exactly in the middle position

rearrange elements into:
elements ≤ x | elements > x

Compute the final destination of every element independently using parallel prefix sums.

We make several parallel passes:
Determine for each element whether it goes left or right
Compute prefix sums of those decisions
Use prefix sums to compute final positions
Scatter elements into an auxiliary array

Step 1: Classify Elements
Create two arrays:
L[i] = 1 if A[i] ≤ x else 0
R[i] = 1 if A[i] > x else 0

Step 2: Parallel Prefix Sums
Compute:
PL = prefix_sum(L)
PR = prefix_sum(R)
where:
PL[i] = number of elements ≤ x before index i
PR[i] = number of elements > x before index i

Step 3: Compute Final Positions
k = total number of elements ≤ x
k = PL[n]

If:
A[i] ≤ x
then its output index is: PL[i]
Otherwise: k + PR[i]

Example

A = [9, 3, 7, 1, 8, 2]
pivot = 5

Classification:
L = [0,1,0,1,0,1]
R = [1,0,1,0,1,0]

Prefix sums:
PL = [0,0,1,1,2,2]
PR = [0,1,1,2,2,3]

Total left count: k = 3

| Element | Destination |
| ------- | ----------- |
| 9       | 3           |
| 3       | 0           |
| 7       | 4           |
| 1       | 1           |
| 8       | 5           |
| 2       | 2           |

Output:
[3,1,2,9,7,8]

Correctly partitioned.
all left elements ≤ pivot
pivot
all right elements ≥ pivot

So these are all valid partitions around pivot 5:
[1,2,3,5,7,8,9]
[3,1,2,5,9,7,8]
[2,3,1,5,8,9,7]

All are correct because:
every left element is < 5
every right element is > 5
5 is in its final sorted position

After partitioning around pivot p:
pivot is guaranteed to be in its final sorted location
recursion only needs to sort:
left partition
right partition

The relative order inside either side is irrelevant.
"""

'\nrearrange elements into:\nelements ≤ x | elements > x\n\nCompute the final destination of every element independently using parallel prefix sums.\n\nWe make several parallel passes:\nDetermine for each element whether it goes left or right\nCompute prefix sums of those decisions\nUse prefix sums to compute final positions\nScatter elements into an auxiliary array\n\nStep 1: Classify Elements\nCreate two arrays:\nL[i] = 1 if A[i] ≤ x else 0\nR[i] = 1 if A[i] > x else 0\n\nStep 2: Parallel Prefix Sums\nCompute:\nPL = prefix_sum(L)\nPR = prefix_sum(R)\nwhere:\nPL[i] = number of elements ≤ x before index i\nPR[i] = number of elements > x before index i\n\nStep 3: Compute Final Positions\nk = total number of elements ≤ x\nk = PL[n]\n\nIf:\nA[i] ≤ x\nthen its output index is: PL[i]\nOtherwise: k + PR[i]\n'

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def prefix_sum(arr):
    out = [0] * len(arr)

    s = 0
    for i in range(len(arr)):
        out[i] = s
        s += arr[i]

    return out, s


def parallel_partition(A, pivot):
    n = len(A)
    L = [0] * n
    R = [0] * n

    def classify(i):
        if A[i] <= pivot:
            L[i] = 1
        else:
            R[i] = 1

    with ThreadPoolExecutor() as ex:
        ex.map(classify, range(n))

    PL, left_count = prefix_sum(L)
    PR, right_count = prefix_sum(R)

    B = [None] * n

    def scatter(i):
        if A[i] <= pivot:
            pos = PL[i]
        else:
            pos = left_count + PR[i]

        B[pos] = A[i]

    with ThreadPoolExecutor() as ex:
        ex.map(scatter, range(n))

    return B

In [ ]:
"""
FFT of even coefficients is independent
FFT of odd coefficients is independent

| Measure     | Complexity        |
| ----------- | ------------------|
| Work        | (O(nlog n))       |
| Span        | (O(log n))        |
| Parallelism | (O(n))            |
| Extra space | (O(nlog n)) naive |


cmath:
complex numbers
Euler's formula
roots of unity
complex exponentials

which are essential for FFT.
"""
import cmath
from concurrent.futures import ThreadPoolExecutor

CUTOFF = 32

def serial_fft(a):
    n = len(a)

    if n == 1:
        return a

    even = serial_fft(a[0::2])
    odd = serial_fft(a[1::2])

    y = [0] * n

    for k in range(n // 2):
        w = cmath.exp(-2j * cmath.pi * k / n)

        t = w * odd[k]

        y[k] = even[k] + t
        y[k + n // 2] = even[k] - t

    return y

def parallel_fft(a, executor=None):
    n = len(a)

    if n <= CUTOFF:
        return serial_fft(a)

    even_part = a[0::2]
    odd_part = a[1::2]

    # Parallel recursive FFTs
    if executor is None:
        with ThreadPoolExecutor() as ex:
            future_even = ex.submit(parallel_fft, even_part, ex)
            odd = parallel_fft(odd_part, ex)
            even = future_even.result()
    else:
        future_even = executor.submit(
            parallel_fft,
            even_part,
            executor
        )

        odd = parallel_fft(odd_part, executor)
        even = future_even.result()

    y = [0] * n

    # Parallel butterfly
    def butterfly(k):
        w = cmath.exp(-2j * cmath.pi * k / n)

        t = w * odd[k]

        y[k] = even[k] + t
        y[k + n // 2] = even[k] - t

    if executor is None:
        with ThreadPoolExecutor() as ex:
            list(ex.map(butterfly, range(n // 2)))
    else:
        list(executor.map(butterfly, range(n // 2)))

    return y

In [18]:
"""
Think of FFT as Compression of Computation
Instead of recomputing similar terms repeatedly, FFT shares work recursively.
That is why the butterfly operations exist.

a ------\      /---- a + wb
          \    /
           \  /
           /  \
          /    \
b ------/      \---- a - wb

That crossed pattern resembles butterfly wings.

The butterfly computes:

y[k] = even[k] + w ⋅ odd[k]
y[k+n/2] = even[k] − w ⋅ odd[k]
w is a root of unity
one input pair produces two outputs

Example Butterfly
Suppose:
even[k] = 4
odd[k] = 6
w = 1

Then:
t = w * odd[k] = 6

Outputs:
y[k]       = 4 + 6 = 10
y[k+n/2]   = 4 - 6 = -2

For n=8, the FFT network contains many butterflies:
Stage 1: small butterflies
Stage 2: medium butterflies
Stage 3: large butterflies
The entire FFT looks like an interconnected butterfly network.

So one pair becomes two transformed outputs.

Instead of recomputing polynomial evaluations independently,
the FFT shares intermediate computations through these butterfly combinations.

For 2 numbers:
FFT = worse

For millions of coefficients:
FFT = vastly better

Exactly like:
merge sort is slower than insertion sort for tiny arrays
but much faster for huge arrays

Why do we transform the data into these weird complex numbers?
The FFT converts data from the time/value domain into the frequency domain.

Those complex numbers encode:
frequencies
amplitudes
phase information

Because many operations become MUCH easier in frequency space.

Polynomial Multiplication

Suppose:
A(x),B(x)

are polynomials.
Naively multiplying them costs:
O(n^2)

But FFT lets us do:
FFT of A
FFT of B
Multiply pointwise
Inverse FFT

Total:
O(nlogn)
This is huge.



Roots of unity means:
Numbers x such that:
x^n  = 1

For the 4th roots of unity, we solve:
x^4 = 1

The solutions are:
1, i, −1, −i

These are complex numbers evenly spaced around the unit circle.

i = sqrt(−1)
i^2 = −1

i^4 = (i^2)^2 = (−1)^2 = 1

This creates points on the unit circle.
e^iθ = cos(θ) + isin(θ)

cmath.exp(-2j * pi * k / n) = e^(−2πik/n)
which rotates around the circle.



4 seems to work out perfectly around unit circle (360°/4 = 90°)

Say we had an array of 9 elements:
The 9 roots are still evenly spaced around the unit circle.
Instead of 90° jumps like n=4, you get:

360°/9=40° spacing.
So each successive root rotates by 40°.


The FFT itself is not the final goal.
The goal is:
Fast convolution.
Polynomial multiplication is convolution.

The Hidden Win
The "expensive-looking" FFT transform is carefully structured so that it recursively reuses computations.

That's where the speedup comes from.


A Better Intuition

The FFT is analogous to:
Matrix Multiplication

Naive matrix multiplication:

O(n^3)

Strassen:
O(n^2.81)

For tiny matrices:
Strassen is worse

For huge matrices:
Strassen wins

Same story with FFT.
"""

'\nRoots of unity means:\n\nNumbers x such that:\nx^n  = 1\n\nFor the 4th roots of unity, we solve:\nx^4 = 1\n\nThe solutions are:\n1, i, −1, −i\n\nThese are complex numbers evenly spaced around the unit circle.\n\ni = sqrt(−1)\ni^2 = −1\n\ni^4 = (i^2)^2 = (−1)^2 = 1\n\nThis creates points on the unit circle.\ne^iθ = cos(θ) + isin(θ)\n\ncmath.exp(-2j * pi * k / n) = e^(−2πik/n)\nwhich rotates around the circle.\n\n\n\n4 seems to work out perfectly around unit circle (360°/4 = 90°)\n\nSay we had an array of 9 elements:\nThe 9 roots are still evenly spaced around the unit circle.\nInstead of 90° jumps like n=4, you get:\n\n360°/9=40° spacing.\nSo each successive root rotates by 40°.\n'

In [ ]:
"""
Parallel SELECT:

| Measure     | Complexity    |
| ----------- | --------------|
| Work        | (O(n))        |
| Span        | (O(log^2 n))  |
| Parallelism | (O(nlog^2 n)) |


Why Groups of 5 Matter
This guarantee depends critically on group size.
Smaller groups weaken the bound.

For example:
groups of 3 do NOT guarantee linear time
groups of 5 do

because the discarded fraction becomes large enough.
"""
from concurrent.futures import ThreadPoolExecutor

CUTOFF = 32

def serial_select(A, k):
    return sorted(A)[k]

def median_of_five(group):
    return sorted(group)[len(group) // 2]

def parallel_partition(A, pivot):
    L = []
    E = []
    G = []

    for x in A:
        if x < pivot:
            L.append(x)
        elif x > pivot:
            G.append(x)
        else:
            E.append(x)

    return L, E, G


def p_select(A, k, executor=None):
    n = len(A)

    if n <= CUTOFF:
        return serial_select(A, k)

    # groups of 5
    groups = [
        A[i:i+5]
        for i in range(0, n, 5)
    ]

    # compute medians in parallel
    if executor is None:
        with ThreadPoolExecutor() as ex:
            medians = list(
                ex.map(median_of_five, groups)
            )
            pivot = p_select(
                medians,
                len(medians)//2,
                ex
            )
    else:
        medians = list(
            executor.map(median_of_five, groups)
        )

        pivot = p_select(
            medians,
            len(medians)//2,
            executor
        )

    # partition
    L, E, G = parallel_partition(A, pivot)

    # recurse
    if k < len(L):
        return p_select(L, k, executor)

    elif k < len(L) + len(E):
        return pivot

    else:
        return p_select(
            G,
            k - len(L) - len(E),
            executor
        )

In [ ]:
"""
serial pseudocode
def sum_arrays(A, B, C, n): 
    parallel for i = 1 to n:
        C[i] = A[i] + B[i]
"""
from concurrent.futures import ThreadPoolExecutor

def sum_arrays_recursive(A, B, C, low, high, executor):
    if low == high:
        C[low] = A[low] + B[low]
        return

    mid = (low + high) // 2

    future = executor.submit(
        sum_arrays_recursive,
        A, B, C,
        low, mid,
        executor
    )

    sum_arrays_recursive(A, B, C, mid + 1, high, executor)
    future.result()

def sum_arrays(A, B):
    n = len(A)
    C = [0] * n

    with ThreadPoolExecutor() as executor:
        sum_arrays_recursive(A, B, C, 0, n - 1, executor)

    return C

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import math


def add_subarray(A, B, C, start, end):
    for k in range(start, end + 1):
        C[k] = A[k] + B[k]


def sum_arrays_d(A, B, grain_size):
    n = len(A)
    C = [0] * n

    # Number of chunks
    r = math.ceil(n / grain_size)

    with ThreadPoolExecutor() as executor:
        futures = []

        for k in range(r):
            start = k * grain_size
            end = min((k + 1) * grain_size - 1, n - 1)

            future = executor.submit(
                add_subarray,
                A, B, C,
                start, end
            )

            futures.append(future)

        for future in futures:
            future.result()

    return C


# Example
# A = [1, 2, 3, 4, 5, 6, 7, 8]
# B = [10, 20, 30, 40, 50, 60, 70, 80]

# result = sum_arrays_d(A, B, grain_size=2)

# print(result)
# [11, 22, 33, 44, 55, 66, 77, 88]

In [19]:
"""
Why does grain_size improve the algorithm?

The idea is to reduce parallel overhead.

If grain_size = 1:
one task is created per array element
total tasks = n

So for a huge array:
n = 1,000,000

you create 1 million parallel tasks.

That is usually inefficient because:
task creation costs time
synchronization costs time
scheduling costs time
context switching costs time

The overhead can become larger than the actual addition operation.

What grain size does
Instead of:
1 task -> 1 addition

we use:
1 task -> many additions

| Small grain size      | Large grain size     |
| --------------------- | -------------------- |
| More parallelism      | Less overhead        |
| More task overhead    | Fewer tasks          |
| Better load balancing | Worse load balancing |

smaller g → more parallelism
larger g → less overhead

Real systems choose g to balance both effects.


Example:
grain_size = 1000

Now each task handles 1000 elements.

Number of tasks becomes:
n/1000
instead of n.

fewer tasks
lower scheduling overhead
better cache usage
better CPU efficiency
"""

'\nWhy does grain_size improve the algorithm?\n\nThe idea is to reduce parallel overhead.\n\nIf grain_size = 1:\none task is created per array element\ntotal tasks = n\n\nSo for a huge array:\nn = 1,000,000\n\nyou create 1 million parallel tasks.\n\nThat is usually inefficient because:\ntask creation costs time\nsynchronization costs time\nscheduling costs time\ncontext switching costs time\n\nThe overhead can become larger than the actual addition operation.\n\nWhat grain size does\nInstead of:\n1 task -> 1 addition\n\nwe use:\n1 task -> many additions\n\n| Small grain size      | Large grain size     |\n| --------------------- | -------------------- |\n| More parallelism      | Less overhead        |\n| More task overhead    | Fewer tasks          |\n| Better load balancing | Worse load balancing |\n\nsmaller g → more parallelism\nlarger g → less overhead\n\nReal systems choose g to balance both effects.\n\n\nExample:\ngrain_size = 1000\n\nNow each task handles 1000 elements.\n\nNumb

In [20]:
"""
p_matrix_multiply_recursive

multiplying an nxn matrices parallel without creating a temp C
by: C += A * B from different quadriants topleft topright/bottomleft bottomright
The trade-off is generally worthwhile because the original algorithm already exposes more parallelism
than real hardware can exploit, while the modified version significantly reduces memory overhead.
"""

'\np_matrix_multiply_recursive\n\nmultiplying an nxn matrices parallel without creating a temp C\nby: C += A * B from different quadriants topleft topright/bottomleft bottomright\nThe trade-off is generally worthwhile because the original algorithm already exposes more parallelism\nthan real hardware can exploit, while the modified version significantly reduces memory overhead.\n'

In [ ]:
"""
Parallel LU decomposition (without pivoting)
"""
from concurrent.futures import ThreadPoolExecutor
import numpy as np


def eliminate_row(A, k, i, n):
    A[i, k] = A[i, k] / A[k, k]

    for j in range(k + 1, n):
        A[i, j] -= A[i, k] * A[k, j]


def parallel_lu(A):
    A = A.astype(float)
    n = A.shape[0]

    with ThreadPoolExecutor() as executor:
        for k in range(n):
            futures = []

            for i in range(k + 1, n):
                futures.append(
                    executor.submit(
                        eliminate_row,
                        A, k, i, n
                    )
                )

            for f in futures:
                f.result()

    return A

In [ ]:
"""
Parallel LUP Decomposition

LUP adds:
pivot search,
row swaps.

The elimination phase remains parallelizable.
"""
from concurrent.futures import ThreadPoolExecutor
import numpy as np


def eliminate_row_lup(A, k, i, n):
    A[i, k] /= A[k, k]

    for j in range(k + 1, n):
        A[i, j] -= A[i, k] * A[k, j]


def parallel_lup(A):
    A = A.astype(float)
    n = A.shape[0]
    P = np.arange(n)

    with ThreadPoolExecutor() as executor:
        for k in range(n):
            # Pivot search (sequential)
            pivot = np.argmax(np.abs(A[k:n, k])) + k

            if A[pivot, k] == 0:
                raise ValueError("Singular matrix")

            if pivot != k:
                A[[k, pivot]] = A[[pivot, k]]
                P[[k, pivot]] = P[[pivot, k]]

            futures = []

            for i in range(k + 1, n):
                futures.append(
                    executor.submit(
                        eliminate_row_lup,
                        A, k, i, n
                    )
                )

            for f in futures:
                f.result()

    return P, A

In [ ]:
"""
Parallel LUP-SOLVE

LUP solve performs:
forward substitution,
backward substitution.

Unfortunately both are inherently sequential because each variable depends on previous variables.
We can only parallelize inner dot products.
"""
import numpy as np
from concurrent.futures import ThreadPoolExecutor


def parallel_dot(a, b):
    with ThreadPoolExecutor() as executor:
        results = list(executor.map(lambda x: x[0] * x[1], zip(a, b)))

    return sum(results)

def lup_solve(LU, P, b):
    n = len(b)
    x = np.zeros(n)
    y = np.zeros(n)
    pb = b[P]

    # Forward substitution
    for i in range(n):

        y[i] = pb[i] - parallel_dot(
            LU[i, :i],
            y[:i]
        )

    # Back substitution
    for i in reversed(range(n)):

        x[i] = (
            y[i] -
            parallel_dot(
                LU[i, i + 1:],
                x[i + 1:]
            )
        ) / LU[i, i]

    return x

In [ ]:
"""
Parallel Cholesky decomposition
"""
import numpy as np
from concurrent.futures import ThreadPoolExecutor

def update_column(L, A, k, i):
    s = np.dot(L[i, :k], L[k, :k])
    L[i, k] = (A[i, k] - s) / L[k, k]


def parallel_cholesky(A):
    n = A.shape[0]
    L = np.zeros_like(A, dtype=float)

    with ThreadPoolExecutor() as executor:
        for k in range(n):
            # Diagonal computation
            s = np.dot(L[k, :k], L[k, :k])
            L[k, k] = np.sqrt(A[k, k] - s)
            futures = []

            for i in range(k + 1, n):
                futures.append(
                    executor.submit(
                        update_column,
                        L, A, k, i
                    )
                )

            for f in futures:
                f.result()

    return L

In [ ]:
"""
Matrix inversion using Cholesky
"""
def invert_spd(A):
    L = parallel_cholesky(A)
    n = A.shape[0]
    I = np.eye(n)
    A_inv = np.zeros_like(A)

    # Solve AX = I column-by-column
    with ThreadPoolExecutor() as executor:
        futures = []

        for j in range(n):
            futures.append(
                executor.submit(
                    np.linalg.solve,
                    A,
                    I[:, j]
                )
            )

        for j, f in enumerate(futures):
            A_inv[:, j] = f.result()

    return A_inv

In [21]:
"""
| Algorithm              | Work          | Span        | Parallelism|
| ---------------------- | ------------- | ------------| -----------|
| Parallel LU            | (Theta(n^3)) | (Theta(n^2)) | (Theta(n)) |
| Parallel LUP           | (Theta(n^3)) | (Theta(n^2)) | (Theta(n)) |
| Parallel LUP-SOLVE     | (Theta(n^2)) | (Theta(n))   | (Theta(n)) |
| Parallel SPD inversion | (Theta(n^3)) | (Theta(n^2)) | (Theta(n)) |
"""

'\n| Algorithm              | Work          | Span        | Parallelism|\n| ---------------------- | ------------- | ------------| -----------|\n| Parallel LU            | (Theta(n^3)) | (Theta(n^2)) | (Theta(n)) |\n| Parallel LUP           | (Theta(n^3)) | (Theta(n^2)) | (Theta(n)) |\n| Parallel LUP-SOLVE     | (Theta(n^2)) | (Theta(n))   | (Theta(n)) |\n| Parallel SPD inversion | (Theta(n^3)) | (Theta(n^2)) | (Theta(n)) |\n'

In [22]:
"""
# ============================================================
# LU, LUP, LUP-SOLVE, and Cholesky Examples
# ============================================================

# These algorithms solve systems of equations:
#
#     Ax = b
#
# where:
#   A = coefficient matrix
#   x = unknown variables
#   b = outputs/constants
#
# Example:
#
#   2x + y = 5
#   4x + 3y = 11
#
# Matrix form:
#
#   [2 1] [x] = [5 ]
#   [4 3] [y]   [11]
#
# ============================================================
# 1. LU DECOMPOSITION
# ============================================================

# LU factors a matrix into:
#
#   A = L U
#
# where:
#
#   L = lower triangular
#   U = upper triangular
#
# Why?
#
# Solving triangular systems is much easier than solving
# the original matrix directly.
#
# ============================================================

import numpy as np
from scipy.linalg import lu

A = np.array([
    [2., 1.],
    [4., 3.]
])

P, L, U = lu(A)

print("========== LU DECOMPOSITION ==========")

print("\nOriginal matrix A:")
print(A)

print("\nLower triangular matrix L:")
print(L)

print("\nUpper triangular matrix U:")
print(U)

print("\nL @ U:")
print(L @ U)

# What happened mathematically:
#
# Start:
#
#   [2 1]
#   [4 3]
#
# Eliminate the 4 below pivot 2:
#
# Row2 = Row2 - 2*Row1
#
# Result:
#
#   [2 1]
#   [0 1]
#
# This becomes U.
#
# The multiplier 2 is stored in L.


# ============================================================
# 2. LUP DECOMPOSITION
# ============================================================

# LU can fail if a pivot is zero.
#
# Example:
#
#   [0 1]
#   [2 3]
#
# First pivot = 0 -> cannot divide.
#
# Solution:
# swap rows first.
#
# This introduces permutation matrix P:
#
#   P A = L U
#
# ============================================================

A2 = np.array([
    [0., 1.],
    [2., 3.]
])

P2, L2, U2 = lu(A2)

print("\n\n========== LUP DECOMPOSITION ==========")

print("\nOriginal matrix A:")
print(A2)

print("\nPermutation matrix P:")
print(P2)

print("\nLower triangular matrix L:")
print(L2)

print("\nUpper triangular matrix U:")
print(U2)

print("\nP @ A:")
print(P2 @ A2)

print("\nL @ U:")
print(L2 @ U2)

# P records row swaps.
#
# This improves:
#   - stability
#   - numerical accuracy
#   - avoids divide-by-zero


# ============================================================
# 3. LUP SOLVE
# ============================================================

# After decomposition:
#
#   P A = L U
#
# solving becomes easy.
#
# Step 1:
#
#   L y = P b
#
# (forward substitution)
#
# Step 2:
#
#   U x = y
#
# (backward substitution)
#
# ============================================================

from scipy.linalg import lu_factor, lu_solve

A3 = np.array([
    [2., 1.],
    [4., 3.]
])

b = np.array([5., 11.])

lu_matrix, pivots = lu_factor(A3)

x = lu_solve((lu_matrix, pivots), b)

print("\n\n========== LUP SOLVE ==========")

print("\nMatrix A:")
print(A3)

print("\nVector b:")
print(b)

print("\nSolution x:")
print(x)

print("\nCheck A @ x:")
print(A3 @ x)

# Why this matters:
#
# If many systems share the same A:
#
#   A x1 = b1
#   A x2 = b2
#   A x3 = b3
#
# we decompose A ONCE and solve cheaply many times.


# ============================================================
# 4. SPD MATRICES + CHOLESKY DECOMPOSITION
# ============================================================

# SPD = Symmetric Positive Definite
#
# Symmetric:
#
#   A = A^T
#
# Positive definite:
#
#   x^T A x > 0
#
# Example:
#
#   [4 2]
#   [2 2]
#
# SPD matrices allow faster decomposition:
#
#   A = L L^T
#
# called Cholesky decomposition.
#
# ============================================================

A4 = np.array([
    [4., 2.],
    [2., 2.]
])

L4 = np.linalg.cholesky(A4)

print("\n\n========== CHOLESKY DECOMPOSITION ==========")

print("\nSPD matrix A:")
print(A4)

print("\nLower triangular matrix L:")
print(L4)

print("\nL @ L.T:")
print(L4 @ L4.T)

# Why Cholesky is important:
#
# Compared to LU:
#   - fewer operations
#   - more stable
#   - less memory
#   - faster
#
# Used heavily in:
#   - optimization
#   - machine learning
#   - statistics
#   - covariance matrices


# ============================================================
# 5. MATRIX INVERSE
# ============================================================

# Matrix inverse satisfies:
#
#   A A^-1 = I
#
# Then:
#
#   x = A^-1 b
#
# ============================================================

A5 = np.array([
    [4., 2.],
    [2., 2.]
])

A_inv = np.linalg.inv(A5)

print("\n\n========== MATRIX INVERSE ==========")

print("\nMatrix A:")
print(A5)

print("\nInverse of A:")
print(A_inv)

print("\nA @ A_inv:")
print(A5 @ A_inv)

# IMPORTANT:
#
# In practice we usually DO NOT compute inverses directly.
#
# Better:
#   decompose matrix once
#   solve systems directly
#
# because:
#   - faster
#   - more numerically stable
#   - less floating-point error


# ============================================================
# SUMMARY
# ============================================================

LU: General matrix decomposition
LUP: LU + row swaps for stability
LUP-SOLVE: Solve Ax=b efficiently after decomposition
Cholesky: Faster decomposition for SPD matrices
Inverse: Explicitly compute A^-1
"""

'\n# ============================================================\n# LU, LUP, LUP-SOLVE, and Cholesky Examples\n# ============================================================\n\n# These algorithms solve systems of equations:\n#\n#     Ax = b\n#\n# where:\n#   A = coefficient matrix\n#   x = unknown variables\n#   b = outputs/constants\n#\n# Example:\n#\n#   2x + y = 5\n#   4x + 3y = 11\n#\n# Matrix form:\n#\n#   [2 1] [x] = [5 ]\n#   [4 3] [y]   [11]\n#\n# ============================================================\n# 1. LU DECOMPOSITION\n# ============================================================\n\n# LU factors a matrix into:\n#\n#   A = L U\n#\n# where:\n#\n#   L = lower triangular\n#   U = upper triangular\n#\n# Why?\n#\n# Solving triangular systems is much easier than solving\n# the original matrix directly.\n#\n# ============================================================\n\nimport numpy as np\nfrom scipy.linalg import lu\n\nA = np.array([\n    [2., 1.],\n    [4., 3.]\n])\n\

In [23]:
"""
Suppose we want to solve:

Ax = b

A = |  2   1   1 |
    |  4  -6   0 |
    | -2   7   2 |

b = |  5 |
    | -2 |
    |  9 |

The goal is to find x.

Every algorithm below is essentially trying to solve this problem,
but in increasingly practical ways.

================================================================
1. LU DECOMPOSITION
================================================================

LU factors A into:

A = LU

L = lower triangular
L = | 1  0  0 |
    | *  1  0 |
    | *  *  1 |

U = upper triangular
U = | *  *  * |
    | 0  *  * |
    | 0  0  * |

--------------------------------------------------------------
Step 1: Eliminate below first pivot
--------------------------------------------------------------

A = |  2   1   1 |
    |  4  -6   0 |
    | -2   7   2 |

Pivot = 2

Multiplier for row 2:

m21 = 4/2 = 2

Row2 ← Row2 - 2(Row1)

Multiplier for row 3:

m31 = -2/2 = -1

Row3 ← Row3 + Row1

Result:
| 2   1   1 |
| 0  -8  -2 |
| 0   8   3 |

--------------------------------------------------------------
Step 2: Eliminate below second pivot
--------------------------------------------------------------

Pivot = -8

m32 = 8/(-8) = -1

Row3 ← Row3 + Row2

Result:

U =
| 2   1   1 |
| 0  -8  -2 |
| 0   0   1 |

Store multipliers in L:

L =
| 1   0   0 |
| 2   1   0 |
|-1  -1   1 |

A = LU

The matrix has been factored.
Solving triangular systems is easy.

Instead of:
Ax = b

solve:
LUx = b

Let:
Ux = y

Then:
Ly = b
Ux = y

which are easy.

================================================================
2. LUP DECOMPOSITION
================================================================

What if a pivot is zero?

Example:

A =
| 0   2   1 |
| 4   1   3 |
| 2   5   1 |

The first pivot is 0.
We cannot divide by 0.

LU fails.

--------------------------------------------------------------
Solution: Pivoting
--------------------------------------------------------------

Swap row 1 and row 2:
| 4   1   3 |
| 0   2   1 |
| 2   5   1 |

Now elimination works.

The row swap is stored in P:
PA = LU
P = permutation matrix.

Original system:
Ax = b

becomes:
PAx = Pb

Then:
LUx = Pb

This is why LUP is what real software uses.

================================================================
3. LUP SOLVE
================================================================

Suppose we've already computed:
PA = LU

Now solve:
Ax = b

--------------------------------------------------------------
Step 1
--------------------------------------------------------------

Ly = Pb
This is called forward substitution.

Example:

| 1  0  0 | |y1|   | 5 |
| 2  1  0 | |y2| = |-2 |
|-1 -1  1 | |y3|   | 9 |

Equation 1:
y1 = 5

Equation 2:
2(5) + y2 = -2

y2 = -12

Equation 3:
-5 + 12 + y3 = 9

y3 = 2

So:

y =
|  5 |
|-12 |
|  2 |

--------------------------------------------------------------
Step 2
--------------------------------------------------------------

Solve:

Ux = y

| 2   1   1 | |x1|   |  5 |
| 0  -8  -2 | |x2| = |-12 |
| 0   0   1 | |x3|   |  2 |

Start at bottom:

x3 = 2

Then:

-8x2 - 2(2) = -12

x2 = 1

Then:

2x1 + 1 + 2 = 5

x1 = 1

Answer:

x =
|1|
|1|
|2|

Notice:

LUP-SOLVE depends on LU/LUP.
You must factor first.

Then solve.

================================================================
4. CHOLESKY DECOMPOSITION
================================================================

Suppose

A =
| 4   2   2 |
| 2   5   1 |
| 2   1   3 |

This matrix is:

1. Symmetric

A = A^T

2. Positive Definite

This special structure allows:

A = LL^T

instead of LU.

--------------------------------------------------------------
Result
--------------------------------------------------------------

L =
|2  0  0 |
|1  2  0 |
|1  0  1.414|

and

LL^T = A

--------------------------------------------------------------
Why use Cholesky?
--------------------------------------------------------------

LU works.

But Cholesky uses the symmetry.

Benefits:

- roughly half the work
- less memory
- more stable
- faster

So whenever A is SPD:

Use Cholesky instead of LU.

================================================================
5. MATRIX INVERSE
================================================================

The inverse satisfies:

AA⁻¹ = I

Suppose

A =
|2 0 0|
|0 3 0|
|0 0 4|

Then

A⁻¹ =
|1/2  0    0 |
| 0  1/3   0 |
| 0   0   1/4|

because

A A⁻¹ = I

--------------------------------------------------------------
How is inverse related to solving?
--------------------------------------------------------------

Since:

Ax = b

multiply by A⁻¹:

A⁻¹Ax = A⁻¹b

Ix = A⁻¹b

x = A⁻¹b

So solving equations and computing inverses are related.

However:
In practice we almost never compute A⁻¹.

Instead:
1. Compute LU or Cholesky once.
2. Solve systems directly.

This is faster and more accurate.

================================================================
HOW EVERYTHING FITS TOGETHER
================================================================

Goal:
    Ax = b

Option 1:
    Compute A⁻¹
    x = A⁻¹b

Works, but expensive.

--------------------------------------------------------------

Option 2:
    LU decomposition

    A = LU

    Solve:
        Ly = b
        Ux = y

Much better.

--------------------------------------------------------------

Option 3:
    LUP decomposition

    PA = LU

    Solve:
        Ly = Pb
        Ux = y

Same idea as LU but more robust.

This is what numerical libraries usually do.

--------------------------------------------------------------

Option 4:
    If A is symmetric positive-definite

    A = LLᵀ

    Solve:
        Ly = b
        Lᵀx = y

Even faster than LU.

This is why optimization, statistics,
machine learning, Kalman filters, Gaussian processes,
and least-squares solvers heavily use Cholesky.

================================================================
DEPENDENCY CHAIN
================================================================

LU
  ↓
LUP (LU + pivoting)
  ↓
LUP-SOLVE (uses the LU/LUP factors)
  ↓
Matrix inverse can be built from repeated solves

SPD matrix?
  ↓
Use Cholesky instead of LU/LUP
  ↓
Solve systems even faster
"""

"\nSuppose we want to solve:\n\nAx = b\n\nA = |  2   1   1 |\n    |  4  -6   0 |\n    | -2   7   2 |\n\nb = |  5 |\n    | -2 |\n    |  9 |\n\nThe goal is to find x.\n\nEvery algorithm below is essentially trying to solve this problem,\nbut in increasingly practical ways.\n\n================================================================\n1. LU DECOMPOSITION\n================================================================\n\nLU factors A into:\n\nA = LU\n\nL = lower triangular\nL = | 1  0  0 |\n    | *  1  0 |\n    | *  *  1 |\n\nU = upper triangular\nU = | *  *  * |\n    | 0  *  * |\n    | 0  0  * |\n\n--------------------------------------------------------------\nStep 1: Eliminate below first pivot\n--------------------------------------------------------------\n\nA = |  2   1   1 |\n    |  4  -6   0 |\n    | -2   7   2 |\n\nPivot = 2\n\nMultiplier for row 2:\n\nm21 = 4/2 = 2\n\nRow2 ← Row2 - 2(Row1)\n\nMultiplier for row 3:\n\nm31 = -2/2 = -1\n\nRow3 ← Row3 + Row1\n\nResult:\n| 2 

In [ ]:
"""
To compute the reduction in parallel, use a divide-and-conquer strategy that recursively
reduces the left and right halves of the array and then combines the two results.
x[i] ⊗ x[i+1] ⊗ ⋯ ⊗ x[j] = (x[i] ⊗ ⋯ ⊗ x[m]) ⊗ (x[m+1] ⊗ ⋯ ⊗x[j])



from operator import add

def p_reduce_parallel(x, op):
    n = len(x)

    if n == 1:
        return x[0]

    mid = n // 2

    with ProcessPoolExecutor() as executor:
        left_future = executor.submit(
            p_reduce_parallel, x[:mid], op
        )

        right = p_reduce_parallel(x[mid:], op)

        left = left_future.result()   # sync

    return op(left, right)


if __name__ == "__main__":
    arr = [1, 2, 3, 4, 5, 6, 7, 8]
    print(p_reduce_parallel(arr, add))
"""

In [ ]:
# SERIAL:
def p_reduce(x, i, j, op):
    if i == j:
        return x[i]

    m = (i + j) // 2

    left = p_reduce(x, i, m, op)
    right = p_reduce(x, m + 1, j, op)

    return op(left, right)
"""
The actual Python ProcessPoolExecutor version demonstrates the idea,
but creating a new process pool at every recursive call is not efficient in practice.
Parallel runtimes such as Cilk, OpenMP tasks, TBB, Rayon, or ForkJoinPool are designed
specifically for this recursive spawning pattern.
"""
# PARALLEL:
from concurrent.futures import ProcessPoolExecutor
from operator import add

def p_reduce_parallel(x, op):
    n = len(x)

    if n == 1:
        return x[0]

    mid = n // 2

    with ProcessPoolExecutor() as executor:
        left_future = executor.submit(
            p_reduce_parallel, x[:mid], op
        )

        right = p_reduce_parallel(x[mid:], op)
        left = left_future.result()   # sync

    return op(left, right)

arr = [1, 2, 3, 4, 5, 6, 7, 8]
print(p_reduce_parallel(arr, add))

In [24]:
"""
| Measure     | Value            |
| ----------- | -----------------|
| Work        | (Theta(n^2))     |
| Span        | (Theta(lg n))    |
| Parallelism | (Theta(n^2lg n)) |
"""
# SERIAL:
def scan(x):
    n = len(x)

    if n == 0:
        return []

    y = [None] * n
    y[0] = x[0]

    for i in range(1, n):
        y[i] = y[i - 1] + x[i] # replace + with ⨂

    return y

# Parallel:
def p_reduce(x, i, j):
    if i == j:
        return x[i]

    m = (i + j) // 2

    left = p_reduce(x, i, m)
    right = p_reduce(x, m + 1, j)

    return left + right # replace + with ⨂

def p_scan_1_aux(x, y, i, j):
    for l in range(i, j + 1):
        y[l] = p_reduce(x, 0, l)

"""
p_scan_1 is considered an inefficient parallel scan.
"""
def p_scan_1(x):
    n = len(x)
    y = [None] * n

    p_scan_1_aux(x, y, 0, n - 1)

    return y

In [25]:
"""
| Measure     | Result                             |
| ----------- | -----------------------------------|
| Correctness | induction using associativity of ⨂ |
| Work        | (Theta(nlog n))                    |
| Span        | (Theta(log n))                     |
| Parallelism | (Theta(n))                         |

"""
def p_scan_2(x):
    n = len(x)
    y = [None] * n

    p_scan_2_aux(x, y, 0, n - 1)

    return y

"""
The spawn, sync, and parallel for are conceptual here. In the analysis we treat them as parallel operations.

Two recursive scans of size n/2
A parallel-for loop performing n/2 updates
"""
def p_scan_2_aux(x, y, i, j):
    if i == j:
        y[i] = x[i]

    else:
        k = (i + j) // 2

        # spawn
        p_scan_2_aux(x, y, i, k)

        # runs in parallel with left half
        p_scan_2_aux(x, y, k + 1, j)

        # sync

        # parallel for l = k+1 to j
        for l in range(k + 1, j + 1):
            y[l] = y[k] + y[l] # replace + with ⨂

"""
| Algorithm  | Work          | Span           |
| ---------- | --------------| ---------------|
| p_scan_1 | (Theta(n^2))    | (Theta(log n)) |
| p_scan_2 | (Theta(nlog n)) | (Theta(log n)) |
"""

'\n| Algorithm  | Work          | Span           |\n| ---------- | --------------| ---------------|\n| p_scan_1 | (Theta(n^2))    | (Theta(log n)) |\n| p_scan_2 | (Theta(nlog n)) | (Theta(log n)) |\n'

In [26]:
def p_scan_3(x):
    n = len(x)

    y = [None] * n
    t = [None] * n

    y[0] = x[0]

    if n > 1:
        p_scan_up(x, t, 1, n - 1)
        p_scan_down(x[0], x, t, y, 1, n - 1)

    return y

def p_scan_up(x, t, i, j):
    if i == j:
        return x[i]

    k = (i + j) // 2

    left = p_scan_up(x, t, i, k)
    t[k] = left

    right = p_scan_up(x, t, k + 1, j)

    return left + right # replace + with ⨂

def p_scan_down(v, x, t, y, i, j):
    if i == j:
        y[i] = v + x[i] # replace + with ⨂

    else:
        k = (i + j) // 2

        p_scan_down(v, x, t, y, i, k)

        p_scan_down(v + t[k], x, t, y, k + 1, j)

In [28]:
"""
Parentheses matching using a +-scan

Map:
( → +1
) → −1

Example:
( ( ) ( ) ) ( )

becomes
1 1 -1 1 -1 -1 1 -1

Perform a prefix-sum scan:
1 2 1 2 1 0 1 0

Condition 1
No prefix becomes negative.
Otherwise a closing parenthesis appears before a matching opening one.

Condition 2
Final sum equals zero.
which means the number of opens equals the number of closes.

You can also think of the prefix sum as tracking the number of unmatched opening parentheses seen so far:
(  (  )  (  )  )  (  )
1  2  1  2  1  0  1  0

1 means one unmatched (
2 means two unmatched (
0 means everything seen so far is balanced

A well-formed parenthesis string must:

Never go below 0 (can't close more than you've opened).
End at 0 (all opens eventually close).

For example:
( ( ) ) ) ( ( )

becomes
1 1 -1 -1 -1 1 1 -1

Prefix sums:
1 2 1 0 -1 0 1 0
"""
def p_scan_4(x):
    for i in range(1, len(x)):
        x[i] = x[i-1] + x[i] # replace + with ⨂

In [29]:
"""
What is a stencil?

A stencil specifies which neighboring cells are used to compute a cell's value.
Think of a 2D grid (matrix). To compute a particular entry, you look at a fixed pattern of nearby entries.

For example:
A[i,j] = A[i−1,j] + A[i,j−1]

uses the cell directly above and directly left.

The pattern
  X
X C

(where C is the current cell) is the stencil.

Simple example

Suppose:
A[i,j] = A[i−1,j] + A[i,j−1]

and the first row and first column are initialized to 1.

Start
1 1 1 1
1 ? ? ?
1 ? ? ?
1 ? ? ?

Compute:
A[2,2] = A[1,2] + A[2,1] = 1+1 = 2
1 1 1 1
1 2 ? ?
1 ? ? ?
1 ? ? ?

Next:

A[2,3] = A[1,3] + A[2,2] = 1+2 = 3
1 1 1 1
1 2 3 ?
1 ? ? ?
1 ? ? ?

Continuing:
1 1 1 1
1 2 3 4
1 3 6 10
1 4 10 20

Each cell depends only on cells above and/or left.

The CLRS LCS algorithm uses the stencil:
c[i,j] = f(c[i−1,j],c[i,j−1],c[i−1,j−1])

Pattern:
X X
X C

Because dependencies only go:

upward
leftward

we get:

      A11
     /   \
   A12   A21
     \   /
      A22

More explicitly:

A11 Depends on nothing outside itself.
Compute first.

A12
Depends on A11.

A21
Depends on A11.

But A12 and A21 do not depend on each other.

Therefore:
compute A11
then compute A12 and A21 in parallel then compute A22

def simple_stencil(A, r1, r2, c1, c2):
    n = r2 - r1 + 1

    if n == 1:
        compute_cell(A, r1, c1)
        return

    mrow = (r1 + r2) // 2
    mcol = (c1 + c2) // 2

    # A11
    simple_stencil(A, r1, mrow, c1, mcol)

    # A12 || A21
    spawn simple_stencil(A, r1, mrow, mcol + 1, c2)
    simple_stencil(A, mrow + 1, r2, c1, mcol)
    sync

    # A22
    simple_stencil(A, mrow + 1, r2, mcol + 1, c2)
"""

"\nWhat is a stencil?\n\nA stencil specifies which neighboring cells are used to compute a cell's value.\nThink of a 2D grid (matrix). To compute a particular entry, you look at a fixed pattern of nearby entries.\n\nFor example:\nA[i,j] = A[i−1,j] + A[i,j−1]\n\nuses the cell directly above and directly left.\n\nThe pattern\n  X\nX C\n\n(where C is the current cell) is the stencil.\n\nSimple example\n\nSuppose:\nA[i,j] = A[i−1,j] + A[i,j−1]\n\nand the first row and first column are initialized to 1.\n\nStart\n1 1 1 1\n1 ? ? ?\n1 ? ? ?\n1 ? ? ?\n\nCompute:\nA[2,2] = A[1,2] + A[2,1] = 1+1 = 2\n1 1 1 1\n1 2 ? ?\n1 ? ? ?\n1 ? ? ?\n\nNext:\n\nA[2,3] = A[1,3] + A[2,2] = 1+2 = 3\n1 1 1 1\n1 2 3 ?\n1 ? ? ?\n1 ? ? ?\n\nContinuing:\n1 1 1 1\n1 2 3 4\n1 3 6 10\n1 4 10 20\n\nEach cell depends only on cells above and/or left.\n\nThe CLRS LCS algorithm uses the stencil:\nc[i,j] = f(c[i−1,j],c[i,j−1],c[i−1,j−1])\n\nPattern:\nX X\nX C\n\nBecause dependencies only go:\n\nupward\nleftward\n\nwe get:\n\n 

In [30]:
"""
The wavefront algorithm gets essentially as close to that limit as the fork-join model allows.

The real dependency structure is by diagonals, not by blocks.
Bottom works better than the above methods

label coordinates:
(1,1) (1,2) (1,3) (1,4)

(2,1) (2,2) (2,3) (2,4)

(3,1) (3,2) (3,3) (3,4)

(4,1) (4,2) (4,3) (4,4)

Notice:
(2,2)

depends only on cells where
i + j < 4

Similarly:
(1,3)

and
(3,1)

also have
i + j = 4

Step 4: Group cells by i+j
Compute:
i+j

for each cell
2 3 4 5
3 4 5 6
4 5 6 7
5 6 7 8

Now look at equal values:
Diagonal 2
(1,1)
Diagonal 3
(1,2)   (2,1)
Diagonal 4
(1,3)   (2,2)   (3,1)
Diagonal 5
(1,4)   (2,3)   (3,2)   (4,1)

These diagonals are called wavefronts.


Why can a whole wavefront run in parallel?

Take two cells on diagonal 5:
(1,4) and (2,3)

Could one depend on the other?
No

Dependencies always go:
up
left
which means they always move to a smaller value of i+j.

For example:
(2,3)

depends on
(1,3) and (2,2)

both of which lie on diagonal 4.
It never depends on another cell on diagonal 5.
Therefore all cells on a diagonal are independent.

Imagine the matrix filling from the top-left corner.

Initially:
X . . .
. . . .
. . . .
. . . .

After first wavefront:
X X . .
X . . .
. . . .
. . . .

After second wavefront:
X X X .
X X . .
X . . .
. . . .

After third wavefront:
X X X X
X X X .
X X . .
X . . .

The completed region expands diagonally like a wave moving across the grid.
Hence the name wavefront.


Concrete numerical example

Using
A[i,j] = A[i−1,j] + A[i,j−1]

with first row/column equal to 1:
Wavefront 1
1 1 1 1
1 ? ? ?
1 ? ? ?
1 ? ? ?

Compute:
A[2,2]=1+1=2

1 1 1 1
1 2 ? ?
1 ? ? ?
1 ? ? ?
Wavefront 2

Now compute simultaneously:
A[2,3] and A[3,2]
A[2,3] = 1+2 = 3
A[3,2] = 2+1 = 3

1 1 1 1
1 2 3 ?
1 3 ? ?
1 ? ? ?
Wavefront 3

Compute simultaneously:
A[2,4],A[3,3],A[4,2]
1 1 1 1
1 2 3 4
1 3 6 ?
1 4 ? ?
"""

'\nThe wavefront algorithm gets essentially as close to that limit as the fork-join model allows.\n\nThe real dependency structure is by diagonals, not by blocks.\nBottom works better than the above methods\n\nlabel coordinates:\n(1,1) (1,2) (1,3) (1,4)\n\n(2,1) (2,2) (2,3) (2,4)\n\n(3,1) (3,2) (3,3) (3,4)\n\n(4,1) (4,2) (4,3) (4,4)\n\nNotice:\n(2,2)\n\ndepends only on cells where\ni + j < 4\n\nSimilarly:\n(1,3)\n\nand\n(3,1)\n\nalso have\ni + j = 4\n\nStep 4: Group cells by i+j\nCompute:\ni+j\n\nfor each cell\n2 3 4 5\n3 4 5 6\n4 5 6 7\n5 6 7 8\n\nNow look at equal values:\nDiagonal 2\n(1,1)\nDiagonal 3\n(1,2)   (2,1)\nDiagonal 4\n(1,3)   (2,2)   (3,1)\nDiagonal 5\n(1,4)   (2,3)   (3,2)   (4,1)\n\nThese diagonals are called wavefronts.\n\n\nWhy can a whole wavefront run in parallel?\n\nTake two cells on diagonal 5:\n(1,4) and (2,3)\n\nCould one depend on the other?\nNo\n\nDependencies always go:\nup\nleft\nwhich means they always move to a smaller value of i+j.\n\nFor example:\n(2,3)\